# Project: Carbon Footprint Prediction (India)
## Notebook 02: Feature Engineering and Target Alignment

[![Author](https://img.shields.io/badge/Author-Prakash%20Ukhalkar-blue.svg)](https://github.com/prakash-ukhalkar) [![GitHub Repository](https://img.shields.io/badge/GitHub-Repo-lightgrey)](https://github.com/prakash-ukhalkar/India-Agri-Carbon-Prediction) [![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![Pandas](https://img.shields.io/badge/Pandas-Latest-green)](https://pandas.pydata.org/) [![NumPy](https://img.shields.io/badge/NumPy-Latest-lightblue)](https://numpy.org/) [![statsmodels](https://img.shields.io/badge/statsmodels-Latest-orange)](https://www.statsmodels.org/) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

**Objective:** Transform consolidated masters into a stationary, feature-rich dataset for ML modeling.

**Introduction:** This notebook focuses on the Feature Engineering phase. We will align the three thematic master files, handle the high dimensionality of the energy features, and engineer "Lag" and "Rolling Average" variables to capture the temporal momentum of carbon emissions. We will specifically focus on the Water-Energy-Food Nexus to create high-impact research features.

### Step 1: Environment Setup & Data Loading
Explain the importance of feature scaling and stationarity in time-series research.
In econometric and ML time-series predictions, ensuring stationarity (constant mean/variance) prevents spurious correlations. Feature engineering (lags, momentum) allows the model to learn from historical trends, which is essential for capturing "policy lag" in agricultural transitions.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DATA_DIR = '../data/processed/'

# Load raw masters
df_emi = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'master_emissions_panel.csv'))
df_ene = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'master_energy_activity.csv'))
df_res = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'master_land_water_resource.csv'))

print("Data Loaded successfully.")
print(f"Emissions: {df_emi.shape}, Energy: {df_ene.shape}, Resources: {df_res.shape}")


Data Loaded successfully.
Emissions: (3660, 20), Energy: (65, 317), Resources: (65, 30)


### Step 2: Target Variable Definition ($CO_2e$)
Define the primary dependent variable by aggregating specific "Element" codes from the Emissions Master (e.g., Emissions (CO2e) from energy and crops).

In [2]:
# Filter emission data for total CO2 equivalent indicators
target_elements = ['Emissions (CO2)', 'Emissions (CH4)', 'Emissions (N2O)']
df_target = df_emi[df_emi['Element'].isin(target_elements)]

# Pivot to get a single total CO2e per year
# We assume 'Value' holds the numeric emission, but we'll sum over all Item/Elements for the Year
df_grouped = df_target.groupby(['Year_Standardized'])['Value'].sum().reset_index()
df_grouped.rename(columns={'Value': 'Target_Total_CO2e'}, inplace=True)
df_grouped.set_index('Year_Standardized', inplace=True)

print("Target variable defined. Shape:", df_grouped.shape)
df_grouped.head()


Target variable defined. Shape: (34, 1)


,Target_Total_CO2e
Year_Standardized,
1990,42019.8269
1991,49711.6670
1992,55589.2972
1993,63824.3676
1994,69061.3874


### Step 3: Dimensionality Reduction (Energy & Activity)
Address the high dimensionality issue by removing features with >30% missing values or zero variance.

In [3]:
# Set Year as index
df_ene.set_index('Year_Standardized', inplace=True)
df_res.set_index('Year_Standardized', inplace=True)

def reduce_dimensions(df, missing_thresh=0.3):
    # Drop completely empty columns
    df = df.dropna(axis=1, how='all')
    
    # Drop columns with >30% missing values
    limit = len(df) * (1 - missing_thresh)
    df = df.dropna(axis=1, thresh=limit)
    
    # Drop zero variance columns (constants)
    numeric_df = df.select_dtypes(include=[np.number])
    var = numeric_df.var()
    zero_var_cols = var[var == 0].index
    df = df.drop(columns=zero_var_cols)
    
    return df

print(f"Energy before: {df_ene.shape}")
df_ene_reduced = reduce_dimensions(df_ene)
print(f"Energy after: {df_ene_reduced.shape}")

print(f"Resources before: {df_res.shape}")
df_res_reduced = reduce_dimensions(df_res)
print(f"Resources after: {df_res_reduced.shape}")


Energy before: (65, 316)
Energy after: (65, 10)
Resources before: (65, 29)
Resources after: (65, 9)


### Step 4: Engineering "Intensity" & "Nexus" Features
Rationale for calculating Carbon Intensity (CO2e / Total Irrigated Area) and Energy Intensity (GWh / Hectare). These metrics standardize absolute growth allowing for fair academic "Impact Analysis." 

In [4]:
# Master merge for engineering
# Master merge for engineering
master_df = df_grouped.copy()
for d in [df_ene_reduced, df_res_reduced]:
    overlap = [c for c in d.columns if c in master_df.columns]
    master_df = master_df.join(d.drop(columns=overlap), how='inner')

# We'll locate proxy columns for Irrigated Area and Energy consumption.
# Since column names can vary, we use a simple heuristic to find matching columns.
area_col = [c for c in master_df.columns if ('Area' in c or 'Irrigat' in c) and c not in ['Area', 'Area Code', 'Area Code (M49)']]
energy_col = [c for c in master_df.columns if 'Electricity' in c or 'Energy' in c]

# Create intensity features if proxies are available
if area_col:
    # Convert to numeric to avoid string concatenation errors
    area_s = pd.to_numeric(master_df[area_col[0]], errors='coerce')
    master_df['Emission_Intensity_Land'] = master_df['Target_Total_CO2e'] / (area_s + 1e-5)
    print(f"Created Emission_Intensity_Land using {area_col[0]}")

if area_col and energy_col:
    area_s = pd.to_numeric(master_df[area_col[0]], errors='coerce')
    ene_s = pd.to_numeric(master_df[energy_col[0]], errors='coerce')
    master_df['Energy_Intensity_Water'] = ene_s / (area_s + 1e-5)
    print(f"Created Energy_Intensity_Water using {energy_col[0]} and {area_col[0]}")

print("Master DF shape after adding nexus features:", master_df.shape)


Master DF shape after adding nexus features: (34, 12)


### Step 5: Temporal Feature Engineering (Lags & Momentum)
Discuss why $t-1$ and $t-2$ lags are necessary to capture the "policy lag" in agricultural transitions. We create 1-year and 2-year lags for the primary emission and energy features, and calculate 3-year rolling means to smooth out climate-induced volatility.

In [5]:
# --- NEW INTERPOLATION STRATEGY START ---
# 1. Fill leading NaNs for Renewable-associated features with 0 (since they didn't historically exist)
renewable_cols = [c for c in master_df.columns if 'Renewable' in c or 'renew' in c.lower()]
for col in renewable_cols:
    first_valid_index = master_df[col].first_valid_index()
    if first_valid_index is not None:
        master_df.loc[:first_valid_index, col] = master_df.loc[:first_valid_index, col].fillna(0)
    else:
        master_df[col] = master_df[col].fillna(0)
        
# 2. Linear Interpolation for all numeric columns to fill gaps smoothly without artificial historical growth
master_df = master_df.interpolate(method='linear', limit_direction='both')
# --- NEW INTERPOLATION STRATEGY END ---

# Filter strictly numeric columns
numeric_cols = master_df.select_dtypes(include=[np.number]).columns

# 1. Engineering Lags (t-1, t-2)
for col in numeric_cols:
    master_df[f'{col}_lag1'] = master_df[col].shift(1)
    master_df[f'{col}_lag2'] = master_df[col].shift(2)

# 2. Engineering Momentum (Rolling 3-year Mean)
for col in numeric_cols:
    master_df[f'{col}_roll3_mean'] = master_df[col].rolling(window=3).mean()

# Drop rows with NaNs introduced by shifts
eng_cols = [c for c in master_df.columns if pd.api.types.is_numeric_dtype(master_df[c])]
master_df.dropna(subset=eng_cols, inplace=True)
print("Shape after Temporal Engineering & NaN drop:", master_df.shape)


Shape after Temporal Engineering & NaN drop: (32, 21)


### Step 6: Stationarity Testing (ADF Test)
Perform the Augmented Dickey-Fuller (ADF) test to check if the time series is stationary (a requirement for many ML algorithms).

In [6]:
def test_stationarity(timeseries, title):
    print(f"--- ADF Test: {title} ---")
    result = adfuller(timeseries, autolag='AIC')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    if result[1] <= 0.05:
        print("=> Series is STATIONARY")
    else:
        print("=> Series is NOT STATIONARY (needs differencing)")
    print("-" * 30)

target = master_df['Target_Total_CO2e']
test_stationarity(target, 'Target_Total_CO2e (Raw)')

# If not stationary, apply first differencing
if adfuller(target)[1] > 0.05:
    master_df['Target_Total_CO2e_diff'] = master_df['Target_Total_CO2e'].diff()
    # Drop the first NaN row created by diff
    master_df.dropna(subset=['Target_Total_CO2e_diff'], inplace=True)
    test_stationarity(master_df['Target_Total_CO2e_diff'], 'Target_Total_CO2e (1st Diff)')


--- ADF Test: Target_Total_CO2e (Raw) ---
ADF Statistic: 0.8386
p-value: 0.9922
=> Series is NOT STATIONARY (needs differencing)
------------------------------
--- ADF Test: Target_Total_CO2e (1st Diff) ---
ADF Statistic: -5.6681
p-value: 0.0000
=> Series is STATIONARY
------------------------------


### Step 7: Final Master Research Dataset (MRD)
Summarize the final feature set ready for Notebook 03 (Modeling).

In [7]:
# Select the active target depending on stationarity (use diff if available)
active_target = 'Target_Total_CO2e_diff' if 'Target_Total_CO2e_diff' in master_df.columns else 'Target_Total_CO2e'

# Show strongest correlations with the target
numeric_corr = master_df.select_dtypes(include=[np.number]).corr()
print(f"Top 10 correlated features with {active_target}:")
print(numeric_corr[active_target].abs().sort_values(ascending=False).head(10))

# Save the final dataset
output_path = os.path.join(PROCESSED_DATA_DIR, 'final_feature_set.csv')
master_df.to_csv(output_path)
print(f"\nFinal Feature Set saved to {output_path} with shape {master_df.shape}")


Top 10 correlated features with Target_Total_CO2e_diff:
Target_Total_CO2e_diff    1.000000
Target_Total_CO2e         0.213376
Value_lag2                0.140524
Value_lag1                0.116910
Value_roll3_mean          0.115025
Year Code_lag1            0.105004
Year Code_lag2            0.105004
Year Code_roll3_mean      0.105004
Year Code                 0.105004
Value                     0.085710
Name: Target_Total_CO2e_diff, dtype: float64

Final Feature Set saved to ../data/processed/final_feature_set.csv with shape (31, 22)


### Key Findings
Based on the feature engineering and target alignment process:
- **Target Definition:** Successfully defined and isolated the primary target variable (`Target_Total_CO2e`) by aggregating CO2, CH4, and N2O emissions.
- **Dimensionality Reduction:** Cleaned the high-dimensional energy and resource datasets by removing columns with over 30% missing values or zero variance, reducing noise for ML models.
- **Feature Engineering:** Created high-impact domain features like `Emission_Intensity_Land` and `Energy_Intensity_Water` to capture the Water-Energy-Food Nexus.
- **Temporal Momentum:** Engineered 1-year and 2-year lags, as well as 3-year rolling averages, to capture policy lag and climate-induced volatility.
- **Stationarity:** Conducted Augmented Dickey-Fuller (ADF) testing to ensure the target variable is stationary, applying first-order differencing where necessary for robust time-series forecasting.

---
### End of Notebook 02 — Feature Engineering and Target Alignment

**Outputs produced:**
- `../data/processed/final_feature_set.csv` — final engineered dataset ready for modeling


**CRITICAL NOTE FOR NOTEBOOK 03:**
The primary modeling target (`Target_Total_CO2e_diff`) has been 1st-order differenced to achieve stationarity. Original absolute levels (`Target_Total_CO2e`) are preserved in the dataset so that the models in Notebook 03 can perform an **inverse transformation**. This is required to output predictions in actual metric tonnage ($\Delta CO_2e$ + $Last\_Actual\_CO_2e$) for the "Results" section of the paper.

**Next step → Notebook 03:** Machine Learning Modeling & Impact Analysis (SHAP). Building baseline and advanced predictive models (Linear Regression, XGBoost) and analyzing feature importance.

<div align="center"><sub>END OF NOTEBOOK 02</sub></div>